<a href="https://colab.research.google.com/github/msancheza1/Blog_innteractivo/blob/main/ST1630_S5_Notebook_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 ST1630 · Semana 5 — Apache Spark en Profundidad
**Universidad EAFIT · Escuela de Ciencias Aplicadas e Ingeniería**
Prof. Andrés Sacre Alzate · `csanch35@eafit.edu.co`

---

### Cómo usar este notebook
| Símbolo | Significa |
|---------|-----------|
| 🎓 | Celda de **demostración** — el profesor ejecuta y explica |
| ✏️ | Celda de **práctica** — tú la completas o modifica |
| 🔍 | Celda de **observación** — ejecuta y analiza el output |
| 💡 | Celda de **reto** — extensión opcional |

**Regla del notebook:** antes de ejecutar cualquier celda de práctica, intenta predecir el output. Escribe tu predicción en el comentario `# TU PREDICCIÓN:` de cada celda.

---


## ⚙️ Setup — Ejecutar primero (una sola vez)
Instala PySpark en Colab y crea los datasets sintéticos del curso.


In [ ]:
# 🎓 CELDA DE SETUP — ejecutar antes de todo
!pip install pyspark delta-spark faker -q

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd
import time

spark = (SparkSession.builder
    .appName("ST1630-S5")
    .config("spark.sql.shuffle.partitions", "8")   # reducido para Colab
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print(f"✅ Spark {spark.version} listo")
print(f"   Particiones de shuffle: {spark.conf.get('spark.sql.shuffle.partitions')}")


In [ ]:
# 🎓 Generar datasets del curso
from pyspark.sql import functions as F
from faker import Faker
import random, json
from datetime import datetime, timedelta
import builtins # Import builtins to access Python's native functions

fake = Faker('es_CO')
random.seed(42)

REGIONES = ['Bogotá', 'Medellín', 'Cali', 'Barranquilla', 'Otro']
CATEGORIAS = ['Electrónica', 'Ropa', 'Alimentos', 'Hogar', 'Deportes']
CANALES = ['app_movil', 'web', 'tienda_fisica']
PESOS_REGION = [0.40, 0.20, 0.15, 0.10, 0.15]

def gen_pedido(i, con_errores=True):
    fecha = datetime(2026, 1, 1) + timedelta(days=random.randint(0, 210))
    cantidad = random.randint(1, 5)
    # Use builtins.round to ensure Python's native round function is used
    precio = builtins.round(random.uniform(15000, 2500000), 2)
    total = builtins.round(cantidad * precio, 2)

    # Errores intencionales para demostrar limpieza en Silver
    if con_errores:
        if i % 50 == 0:
            total = None                           # nulo intencional
        if i % 73 == 0:
            # Use builtins.abs to ensure Python's native abs function is used,
            # but only if total is not None
            if total is not None:
                total = -builtins.abs(total)                    # negativo intencional
        # Formatos de fecha mixtos
        if i % 3 == 0:
            fecha_str = fecha.strftime('%Y-%m-%d')
        else:
            fecha_str = fecha.strftime('%d/%m/%y')
        # Región sin normalizar
        region = random.choices(REGIONES, PESOS_REGION)[0]
        if i % 7 == 0:
            region = region.upper()
        if i % 11 == 0:
            region = ' ' + region + ' '           # espacios extra
    else:
        fecha_str = fecha.strftime('%Y-%m-%d')
        region = random.choices(REGIONES, PESOS_REGION)[0]

    return {
        'pedido_id': f'ORD-2026-{i:08d}',
        'fecha': fecha_str,
        'region': region,
        'categoria': random.choice(CATEGORIAS),
        'producto': fake.catch_phrase()[:40],
        'cantidad': cantidad,
        'precio_unit': precio,
        'total': total,
        'canal': random.choice(CANALES),
        'devuelto': random.random() < 0.08
    }

# Dataset pequeño para demostraciones rápidas en Colab
N_DEMO = 10_000
datos = [gen_pedido(i) for i in range(N_DEMO)]
df_raw = spark.createDataFrame(datos)

print(f"✅ Dataset generado: {df_raw.count():,} filas")
print(f"   Particiones: {df_raw.rdd.getNumPartitions()}")
df_raw.show(5, truncate=False)


---
## 📚 Repaso S3 — MapReduce y el WordCount
*Antes de ver las APIs de Spark, recordemos de dónde venimos.*


In [ ]:
# 🎓 REPASO S3: WordCount en PySpark — el mismo ejemplo del tablero
# Texto de prueba con el mismo ejemplo de clase
texto = spark.createDataFrame(
    [("el gato come el pescado y el gato duerme",),],
    ["texto"]
)

# MAP: explode divide el texto en palabras (una fila por palabra)
# Esto es una NARROW transformation — sin shuffle
palabras = texto.select(
    explode(split(lower(col("texto")), " ")).alias("palabra")
)

# SHUFFLE + REDUCE: groupBy agrupa por clave (WIDE transformation)
conteo = palabras.groupBy("palabra").count().orderBy(col("count").desc())

print("=== WordCount — el mismo ejemplo de S3 ===")
conteo.show()

# Observar el plan: ¿cuántos Exchange (shuffles) hay?
print("\n=== Plan de ejecución ===")
conteo.explain()


### ✏️ Tu turno — Repaso CAP
> **Recuerda de S3:** el `groupBy` es una **wide transformation** que genera un shuffle.
> En términos del Teorema CAP, Spark maneja ese shuffle **en RAM** (decisión AP)
> mientras que MapReduce lo escribía **a disco** (decisión CP).

**Pregunta para reflexionar:**
¿Cuántos `Exchange` (shuffles) ves en el plan anterior?
¿A qué operación corresponde cada uno?

Escribe tu respuesta en la celda Markdown de abajo:


**Mi análisis del plan:**
- Exchange 1: corresponde a _________________ porque _________________
- Exchange 2: corresponde a _________________ porque _________________
- Posición CAP de Spark: _________________ porque _________________


---
## 📚 Repaso S4 — Parquet vs CSV: la diferencia medible
*El benchmark del slide 10 de S4, ahora ejecutable en vivo.*


In [ ]:
# 🎓 REPASO S4: guardar en CSV y Parquet, medir la diferencia
import os, shutil
import builtins # Import builtins to access Python's native functions

# Escribir el dataset en ambos formatos
df_raw.write.mode("overwrite").option("header", True).csv("/tmp/ventas_csv")
df_raw.write.mode("overwrite").parquet("/tmp/ventas_parquet")

# Medir tamaños
def folder_size_mb(path):
    total = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            if not f.startswith('.') and not f.endswith('.crc'):
                total += os.path.getsize(os.path.join(dirpath, f))
    return builtins.round(total / 1024 / 1024, 2)

size_csv = folder_size_mb("/tmp/ventas_csv")
size_pq  = folder_size_mb("/tmp/ventas_parquet")

print(f"📄 CSV:     {size_csv} MB")
print(f"🗜️  Parquet: {size_pq} MB")
print(f"💰 Ratio:   {builtins.round(size_csv/size_pq, 1)}x más ligero en Parquet")
print(f"   (En Athena: pagarías {builtins.round(size_csv/size_pq,1)}x menos con Parquet)")


In [ ]:
# 🔍 Benchmark: misma consulta, formatos distintos
FILTRO = "Bogotá"

# Leer CSV
t0 = time.time()
df_csv = spark.read.option("header", True).csv("/tmp/ventas_csv")
r_csv = df_csv.filter(col("region") == FILTRO).agg(sum("total")).collect()
t_csv = builtins.round(time.time() - t0, 3)

# Leer Parquet
t0 = time.time()
df_pq = spark.read.parquet("/tmp/ventas_parquet")
r_pq = df_pq.filter(col("region") == FILTRO).agg(sum("total")).collect()
t_pq = builtins.round(time.time() - t0, 3)

print(f"⏱️  CSV:     {t_csv}s")
print(f"⚡ Parquet: {t_pq}s")
print(f"   Ratio:   {builtins.round(t_csv/t_pq, 1)}x más rápido Parquet (dataset pequeño)")
print()
print("💡 Nota: con 10K filas en Colab el ratio es mucho menor al ~9x del slide.")
print("   Con 500K filas en EMR sobre S3 el ratio es mucho más claro.")
print("   ¿Por qué? El predicado pushdown necesita suficientes datos")
print("   para que la diferencia de I/O sea significativa.")


### ✏️ Tu turno — Predicado pushdown
Cambia el filtro de `"Bogotá"` a `"Medellín"` en la celda anterior y vuelve a ejecutar.
¿Cambió el ratio? ¿Por qué sí o por qué no?


---
## 🆕 S5 · Concepto 1 — Jerarquía de APIs: RDD → DataFrame → Dataset


In [ ]:
# 🎓 RDD vs DataFrame — el mismo trabajo, diferente nivel de abstracción

# ── RDD (bajo nivel) ──
# Acceder al RDD subyacente del DataFrame
rdd = df_raw.rdd

# Contar por región con RDD (map + reduceByKey)
# Sin Catalyst — lo que escribimos es literalmente lo que ejecuta
t0 = time.time()
conteo_rdd = (rdd
    .filter(lambda r: r['devuelto'] == False)
    .map(lambda r: (r['region'], float(r['total'] or 0)))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1], ascending=False))
resultado_rdd = conteo_rdd.take(5)
t_rdd = builtins.round(time.time() - t0, 3)

print("=== RDD (sin Catalyst) ===")
for r in resultado_rdd:
    print(f"  {r[0]}: {r[1]:,.0f}")
print(f"  Tiempo: {t_rdd}s")
print()

# ── DataFrame (API estándar) ──
# El Catalyst Optimizer optimiza automáticamente
t0 = time.time()
conteo_df = (df_raw
    .filter(col("devuelto") == False)
    .groupBy("region")
    .agg(sum("total").alias("total_ventas"))
    .orderBy(col("total_ventas").desc()))
resultado_df = conteo_df.take(5)
t_df = builtins.round(time.time() - t0, 3)

print("=== DataFrame (con Catalyst) ===")
for r in resultado_df:
    print(f"  {r['region']}: {r['total_ventas']:,.0f}")
print(f"  Tiempo: {t_df}s")
print()
print(f"📊 DataFrame fue {builtins.round(t_rdd/t_df, 1)}x más rápido que RDD")
print("   (Catalyst optimizó el orden de las operaciones automáticamente)")


In [ ]:
# ✏️ PRÁCTICA — ¿Cuándo usar RDD en vez de DataFrame?
# Tarea: el siguiente código usa RDD para procesar texto libre.
# Intenta reescribirlo con DataFrame API.
# ¿Puedes? ¿Por qué sería más difícil?

# Código original en RDD
resenas_rdd = spark.sparkContext.parallelize([
    "Excelente producto llegó rápido",
    "Muy malo se demoró mucho",
    "Regular el empaque estaba dañado",
    "Perfecto cumplió mis expectativas",
    "Pésimo no llegó nunca",
])

sentimiento = resenas_rdd.map(lambda texto: {
    'texto': texto,
    'positivo': any(p in texto.lower() for p in ['excelente','perfecto','rápido']),
    'negativo': any(n in texto.lower() for n in ['malo','pésimo','dañado','nunca'])
})

print("=== Análisis de sentimiento con RDD ===")
for s in sentimiento.collect():
    emoji = "✅" if s['positivo'] else ("❌" if s['negativo'] else "⚪")
    print(f"  {emoji} {s['texto']}")

# TU PREDICCIÓN: ¿podría esto reescribirse fácilmente con DataFrame?
# Escribe aquí tu respuesta:
# _______________


---
## 🆕 S5 · Concepto 2 — Narrow vs Wide Dependencies
*El concepto que explica por qué `groupBy` es costoso y `filter` no lo es.*


In [ ]:
# 🎓 Observar narrow vs wide EN EL PLAN antes de ejecutar

# Pipeline con operaciones mixtas
pipeline = (df_raw
    .filter(col("devuelto") == False)           # NARROW ✅
    .select("region", "categoria", "total")     # NARROW ✅
    .withColumn("total", col("total").cast("double"))  # NARROW ✅
    .groupBy("region", "categoria")             # WIDE ❌ Exchange
    .agg(
        count("*").alias("num_pedidos"),
        sum("total").alias("total_ventas"),
        avg("total").alias("ticket_promedio")
    )
    .orderBy(col("total_ventas").desc()))       # WIDE ❌ Exchange

print("=== PLAN FÍSICO — busca los 'Exchange' ===")
print("Cada Exchange = un shuffle = una frontera de stage")
print("="*60)
pipeline.explain()


In [ ]:
# 🔍 Contar los Exchange en el plan
plan_str = pipeline._jdf.queryExecution().executedPlan().toString()
n_exchange = plan_str.count("Exchange")
print(f"Número de Exchange (shuffles) encontrados: {n_exchange}")
print()
print("¿Coincide con tu predicción de las diapositivas?")
print("- filter, select, withColumn: NARROW → sin Exchange")
print("- groupBy: WIDE → 1 Exchange")
print("- orderBy: WIDE → 1 Exchange adicional")
print(f"Total esperado: 2 Exchange")


In [ ]:
# ✏️ PRÁCTICA — Clasifica y observa
# Para cada par de transformaciones, predice cuál genera más shuffles
# LUEGO ejecuta y verifica con .explain()

# TU PREDICCIÓN antes de ejecutar:
# Pipeline A (filter → groupBy) tiene ___ Exchange(s)
# Pipeline B (groupBy → filter) tiene ___ Exchange(s)

pipeline_a = (df_raw
    .filter(col("categoria") == "Electrónica")  # filtra PRIMERO
    .groupBy("region")
    .count())

pipeline_b = (df_raw
    .groupBy("region", "categoria")              # agrupa TODO
    .count()
    .filter(col("categoria") == "Electrónica"))  # filtra DESPUÉS

print("=== Pipeline A: filter → groupBy ===")
pipeline_a.explain()
plan_a = pipeline_a._jdf.queryExecution().executedPlan().toString()
print(f"Exchange encontrados: {plan_a.count('Exchange')}")

print()
print("=== Pipeline B: groupBy → filter ===")
pipeline_b.explain()
plan_b = pipeline_b._jdf.queryExecution().executedPlan().toString()
print(f"Exchange encontrados: {plan_b.count('Exchange')}")

print()
print("💡 ¿El Catalyst Optimizer los hace equivalentes o no?")
print("   Observa si los planes son iguales o diferentes.")


In [ ]:
# 🔍 Hazlo Ahora 1 — Clasificación completa
# Las mismas 8 operaciones del slide — ahora con evidencia del plan

operaciones = {
    "filter(precio > 100)":       df_raw.filter(col("precio_unit") > 100),
    "select(region, total)":      df_raw.select("region", "total"),
    "groupBy(categoria).count()": df_raw.groupBy("categoria").count(),
    "distinct()":                 df_raw.select("region").distinct(),
    "coalesce(4)":                df_raw.coalesce(4),
    "repartition(16)":            df_raw.repartition(16),
    "orderBy(total)":             df_raw.orderBy("total"),
    "withColumn(iva)":            df_raw.withColumn("iva", col("precio_unit") * 0.19),
}

print(f"{'Operación':<35} {'Particiones':<15} {'Exchange en plan':<20} {'Tipo'}")
print("="*85)
for nombre, df_op in operaciones.items():
    n_parts = df_op.rdd.getNumPartitions()
    plan = df_op._jdf.queryExecution().executedPlan().toString()
    n_exc = plan.count("Exchange")
    tipo = "⚡ WIDE" if n_exc > 0 else "✅ narrow"
    print(f"{nombre:<35} {n_parts:<15} {n_exc:<20} {tipo}")


---
## 🆕 S5 · Concepto 3 — Catalyst Optimizer: las 4 fases
*El mismo código PySpark puede ejecutarse de formas muy distintas. Catalyst decide cómo.*


In [ ]:
# 🎓 Las 4 fases del Catalyst — comparar el plan en cada etapa

query = (df_raw
    .filter(col("devuelto") == False)
    .filter(col("total") > 100_000)
    .select("region", "categoria", "total")
    .groupBy("region")
    .agg(sum("total").alias("ventas_totales")))

print("=== FASE 1: Logical Plan (sin optimizar) ===")
print(query._jdf.queryExecution().logical().toString()[:800])
print()

print("=== FASE 2: Optimized Logical Plan ===")
print("(Catalyst fusionó los dos filter en uno — predicate pushdown)")
print(query._jdf.queryExecution().optimizedPlan().toString()[:800])
print()

print("=== FASE 4: Physical Plan (lo que realmente ejecuta) ===")
query.explain(mode="formatted")


In [ ]:
import builtins # Asegurar que builtins esté disponible aquí, aunque ya se importó en 06187e17

# 🔍 Predicate Pushdown en acción — la optimización más importante
# Mismo resultado, diferente orden de operaciones → Catalyst los hace equivalentes

# Versión "mal escrita" (lógicamente: leer todo, luego filtrar)
v1 = (df_raw
    .select("region", "categoria", "total", "devuelto")
    .groupBy("region")
    .agg(sum("total").alias("total"))
    .filter(col("total") > 0))   # filtrar DESPUÉS de agregar

# Versión "bien escrita" (filtrar antes)
v2 = (df_raw
    .filter(col("total") > 0)    # filtrar ANTES de agrupar
    .select("region", "categoria", "total", "devuelto")
    .groupBy("region")
    .agg(sum("total").alias("total")))

print("Plan V1 (filter después del groupBy):")
plan_v1 = v1._jdf.queryExecution().optimizedPlan().toString()
print(plan_v1[:400])
print()
print("Plan V2 (filter antes del groupBy):")
plan_v2 = v2._jdf.queryExecution().optimizedPlan().toString()
print(plan_v2[:400])
print()

t0 = time.time(); v1.count(); t_v1 = builtins.round(time.time()-t0,3)
t0 = time.time(); v2.count(); t_v2 = builtins.round(time.time()-t0,3)
print(f"⏱️  V1 (filter después): {t_v1}s")
print(f"⏱️  V2 (filter antes):   {t_v2}s")
print()
print("💡 ¿Los planes optimizados son iguales o diferentes?")
print("   Si Catalyst los igualó, el tiempo debería ser similar.")

In [ ]:
# ✏️ PRÁCTICA — Encuentra la optimización que Catalyst hace por ti
# Escribe una consulta "ineficiente" y observa cómo Catalyst la reescribe

# Consulta "ineficiente" (calcula columnas que luego descarta)
ineficiente = (df_raw
    .withColumn("iva",        col("precio_unit") * 0.19)   # columna extra
    .withColumn("precio_eur", col("precio_unit") / 4200)   # columna extra
    .withColumn("precio_usd", col("precio_unit") / 4100)   # columna extra
    .select("region", "total"))                             # las descarta todas

print("=== ¿Catalyst elimina las columnas que no se usan? ===")
print("Busca en el plan si aparecen 'iva', 'precio_eur', 'precio_usd':")
print()
ineficiente.explain(mode="formatted")

plan_text = ineficiente._jdf.queryExecution().executedPlan().toString()
for col_name in ["iva", "precio_eur", "precio_usd"]:
    aparece = col_name in plan_text
    print(f"  '{col_name}' en el plan físico: {'SÍ (Catalyst NO la eliminó)' if aparece else '❌ NO (Catalyst aplicó column pruning ✅)'}")


---
## 🆕 S5 · Concepto 4 — Particionamiento: el tamaño de los trozos importa


In [ ]:
# 🎓 repartition vs coalesce — diferencias observables

df_base = df_raw   # 8 particiones (configuradas en SparkSession)
print(f"Particiones iniciales: {df_base.rdd.getNumPartitions()}")
print()

# repartition — WIDE (siempre shuffle, redistribuye uniformemente)
import time
import builtins # Asegurar que builtins esté disponible para esta celda
t0 = time.time()
df_rep = df_base.repartition(32)
df_rep.count()  # fuerza la ejecución
t_rep = builtins.round(time.time()-t0, 3)
print(f"repartition(32): {df_rep.rdd.getNumPartitions()} particiones | {t_rep}s")

# coalesce — NARROW (sin shuffle, fusiona particiones existentes)
t0 = time.time()
df_coal = df_base.coalesce(4)
df_coal.count()
t_coal = builtins.round(time.time()-t0, 3)
print(f"coalesce(4):     {df_coal.rdd.getNumPartitions()} particiones | {t_coal}s")

# Intento de coalesce a MÁS particiones que las existentes
df_coal_up = df_base.coalesce(100)
print(f"coalesce(100):   {df_coal_up.rdd.getNumPartitions()} particiones | (no aumenta)")
print()
print("💡 coalesce() SOLO puede reducir particiones, nunca aumentar")
print("   repartition() puede hacer ambas cosas (con shuffle)")

In [ ]:
import builtins # Importar builtins para usar la función round nativa de Python

# 🔍 Impacto del número de particiones en un groupBy

def medir_groupby(n_particiones, label):
    df = df_raw.repartition(n_particiones)
    t0 = time.time()
    df.groupBy("region", "categoria").agg(
        count("*"),
        sum("total")
    ).count()
    return builtins.round(time.time() - t0, 3)

configuraciones = [1, 4, 8, 16, 32]
print(f"{'Particiones':<15} {'Tiempo groupBy':<20} {'Observación'}")
print("="*60)
for n in configuraciones:
    t = medir_groupby(n, str(n))
    obs = ""
    if n < 4:
        obs = "❌ Pocos cores activos, trabajo secuencial"
    elif n <= 8:
        obs = "✅ Balance razonable para Colab"
    elif n <= 16:
        obs = "✅ Más paralelismo"
    else:
        obs = "⚠️ Overhead de scheduling (dataset pequeño)"
    print(f"{n:<15} {t:<20} {obs}")

print()
print("💡 En producción sobre EMR con 500K filas:")
print("   4 executors × 8 cores = 32 particiones óptimas")
print("   Regla: apuntar a particiones de 100-200 MB cada una")

In [ ]:
# ✏️ PRÁCTICA — Hazlo Ahora 2 (del slide)
# Predice el resultado de cada escenario ANTES de ejecutar
# Escribe tu predicción en el comentario correspondiente

# Escenario A: coalesce antes del groupBy
# TU PREDICCIÓN: ¿cuántos cores trabajan activamente?
# ___________________
df_a = df_raw.coalesce(4).groupBy("region").count()
plan_a = df_a._jdf.queryExecution().executedPlan().toString()
print(f"A — coalesce(4) → groupBy: {plan_a.count('Exchange')} Exchange(s)")
print(f"    Particiones finales: {df_a.rdd.getNumPartitions()}")

# Escenario B: repartition antes del groupBy
# TU PREDICCIÓN: ¿cuántos shuffles ocurren en total?
# ___________________
df_b = df_raw.repartition(32).groupBy("region").count()
plan_b = df_b._jdf.queryExecution().executedPlan().toString()
print(f"B — repartition(32) → groupBy: {plan_b.count('Exchange')} Exchange(s)")
print(f"    Particiones finales: {df_b.rdd.getNumPartitions()}")

# Escenario C: partitionBy al escribir
# TU PREDICCIÓN: ¿cuántos archivos se crean?
# ___________________
df_raw.write.mode("overwrite").partitionBy("region").parquet("/tmp/partitioned_test")
import os
archivos = [f for f in os.listdir("/tmp/partitioned_test") if not f.startswith('.')]
print(f"C — partitionBy(region): {len(archivos)} carpetas (una por región)")
for a in sorted(archivos)[:6]:
    print(f"    {a}")


---
## 🆕 S5 · Concepto 5 — Pipeline Bronze → Silver con Delta Lake
*El código del Lab 1b — primero en Colab con datos pequeños, luego en EMR con 500K filas.*


In [ ]:
# 🎓 Escribir a Bronze (datos crudos, sin transformar)
# En producción: s3://tu-bucket/bronze/pedidos/
# En Colab:      /tmp/lake/bronze/pedidos/

BRONZE_PATH = "/tmp/lake/bronze/pedidos"
SILVER_PATH = "/tmp/lake/silver/pedidos"
GOLD_PATH   = "/tmp/lake/gold/kpis"

# Bronze: guardar tal como llegó, con todos los errores incluidos
df_raw.write.format("delta").mode("overwrite").save(BRONZE_PATH)

# Verificar que Delta creó el _delta_log
import os
delta_log = os.listdir(f"{BRONZE_PATH}/_delta_log")
print(f"✅ Bronze escrito: {df_raw.count():,} filas")
print(f"   _delta_log contiene: {delta_log}")
print()

# Leer de Bronze y verificar que los errores están ahí
bronze = spark.read.format("delta").load(BRONZE_PATH)
nulos  = bronze.filter(col("total").isNull()).count()
negativos = bronze.filter(col("total") < 0).count()
print(f"   Nulos en 'total':    {nulos} (errores intencionales del sistema fuente)")
print(f"   Negativos en 'total': {negativos} (errores intencionales)")
print(f"   Bronze es inmutable — estos errores se limpian en Silver, no aquí")


In [ ]:
# 🎓 Pipeline Bronze → Silver (todas las transformaciones)
# Conectar con el concepto de narrow vs wide del Concepto 2

bronze = spark.read.format("delta").load(BRONZE_PATH)

# ── TRANSFORMACIONES NARROW (sin shuffle) ──
silver_new = (bronze
    # Descartar filas malformadas (narrow ✅)
    .filter(col("total").isNotNull())
    .filter(col("total") > 0)

    # Normalizar región: mayúsculas + trim (narrow ✅)
    .withColumn("region", upper(trim(col("region"))))

    # Cast de fecha — manejar los dos formatos del dataset (narrow ✅)
    .withColumn("fecha_parsed",
        coalesce(
            to_date(col("fecha"), "yyyy-MM-dd"),
            to_date(col("fecha"), "dd/MM/yy")
        ))
    .filter(col("fecha_parsed").isNotNull())

    # Calcular total correctamente (narrow ✅)
    .withColumn("total_calc",
        (col("cantidad") * col("precio_unit")).cast("double"))

    # Seleccionar solo las columnas que Silver necesita
    .select(
        col("pedido_id"),
        col("fecha_parsed").alias("fecha"),
        col("region"),
        col("categoria"),
        col("producto"),
        col("cantidad"),
        col("precio_unit"),
        col("total_calc").alias("total"),
        col("canal"),
        col("devuelto"),
        lit(datetime.now().strftime('%Y-%m-%d')).alias("_ingested_at")
    ))

# Verificar transformaciones
print("=== Resultado de las transformaciones ===")
n_antes = bronze.count()
n_despues = silver_new.count()
print(f"Filas en Bronze: {n_antes:,}")
print(f"Filas en Silver: {n_despues:,}")
print(f"Filas descartadas: {n_antes - n_despues:,} ({round((n_antes-n_despues)/n_antes*100,1)}%)")
print()
print("Muestra de Silver (datos limpios):")
silver_new.show(5, truncate=False)


In [ ]:
# 🎓 Observar el plan del pipeline Silver
# ¿Cuántos narrow y cuántos wide hay?

print("=== Plan del pipeline Bronze → Silver ===")
print("Busca 'Exchange' para encontrar las wide dependencies")
print("="*60)
silver_new.explain()

plan_silver = silver_new._jdf.queryExecution().executedPlan().toString()
n_exc = plan_silver.count("Exchange")
print(f"\nExchange encontrados: {n_exc}")
print("¿Por qué hay 0 shuffles si hay filter, withColumn, select?")
print("→ Todas son NARROW transformations — cada partición se procesa sola")


In [ ]:
# 🎓 Escribir Silver (primera escritura — modo overwrite para el lab)
# En producción usaríamos MERGE para ingesta incremental
# Aquí usamos overwrite para simplicidad en Colab

silver_new.write.format("delta").mode("overwrite").save(SILVER_PATH)

silver = spark.read.format("delta").load(SILVER_PATH)
print(f"✅ Silver escrito: {silver.count():,} filas")
print()

# Verificar que los errores fueron limpiados
nulos_silver     = silver.filter(col("total").isNull()).count()
negativos_silver = silver.filter(col("total") < 0).count()
print(f"   Nulos en Silver:    {nulos_silver} ✅")
print(f"   Negativos en Silver: {negativos_silver} ✅")
print()
print("=== Time travel de Delta Lake ===")
print("(funciona porque Delta guarda el historial en _delta_log)")
from delta.tables import DeltaTable
dt = DeltaTable.forPath(spark, SILVER_PATH)
dt.history().select("version", "timestamp", "operation").show()


In [ ]:
# ✏️ PRÁCTICA — Silver → Gold: calcular KPIs
# Completa el pipeline de Silver a Gold calculando los KPIs de negocio
# PISTA: groupBy(region, fecha) es WIDE — espera ver Exchange en el plan

silver = spark.read.format("delta").load(SILVER_PATH)

# TODO: Completa el código calculando:
# 1. Ventas totales por región y fecha (sum de total)
# 2. Número de pedidos por región y fecha (count)
# 3. Ticket promedio por región y fecha (avg de total)
# 4. Tasa de devolución por región (avg de devuelto cast a int)

gold = (silver
    .groupBy("region", "fecha")
    .agg(
        sum("total").alias("ventas_totales"),
        count("*").alias("num_pedidos"),
        # TODO: agrega avg de total como ticket_promedio
        # TODO: agrega avg de cast(devuelto, int) como tasa_devolucion
    )
    .orderBy("region", "fecha"))

print("=== Gold Layer — KPIs de negocio ===")
gold.show(10)

print("\n=== Plan de Gold (¿cuántos Exchange?) ===")
gold.explain()

# Escribir Gold
gold.write.format("delta").mode("overwrite").save(GOLD_PATH)
print(f"\n✅ Gold escrito: {gold.count():,} filas (combinaciones región × fecha)")


---
## 🆕 S5 · Bonus — Features de Delta Lake en acción
*Time travel, schema evolution y las estadísticas del _delta_log.*


In [ ]:
# 🎓 Time Travel — leer una versión anterior de la tabla

from delta.tables import DeltaTable

print("=== Historial de Silver ===")
dt_silver = DeltaTable.forPath(spark, SILVER_PATH)
dt_silver.history().select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

# Leer la versión 0 (antes de cualquier cambio)
v0 = spark.read.format("delta").option("versionAsOf", 0).load(SILVER_PATH)
v_actual = spark.read.format("delta").load(SILVER_PATH)

print(f"\nVersión 0: {v0.count():,} filas")
print(f"Actual:    {v_actual.count():,} filas")
print()
print("💡 En producción, el time travel sirve para:")
print("   - Auditoría regulatoria ('¿qué datos había el 3 de agosto?')")
print("   - Recuperación ante errores ('el pipeline corrió con un bug')")
print("   - GDPR: reproducir el estado antes de un borrado")


In [ ]:
# 🎓 Schema Evolution — agregar una columna sin reescribir

# Simular nuevos datos con una columna adicional: metodo_pago
nuevos_datos = spark.createDataFrame([
    ("ORD-2026-99999999", "2026-08-01", "BOGOTÁ", "Electrónica",
     "Producto nuevo", 1, 500000.0, 500000.0, "app_movil", False,
     "2026-08-01", "tarjeta_credito"),
], ["pedido_id", "fecha", "region", "categoria", "producto",
    "cantidad", "precio_unit", "total", "canal", "devuelto",
    "_ingested_at", "metodo_pago"])   # columna nueva

# Sin mergeSchema: esto fallaría
# nuevos_datos.write.format("delta").mode("append").save(SILVER_PATH)

# Con mergeSchema: Delta acepta el nuevo campo
nuevos_datos.write.format("delta")     .mode("append")     .option("mergeSchema", "true")     .save(SILVER_PATH)

print("=== Schema de Silver después de la evolución ===")
silver_evolucionado = spark.read.format("delta").load(SILVER_PATH)
silver_evolucionado.printSchema()
print()
print(f"Filas con metodo_pago no nulo: {silver_evolucionado.filter(col('metodo_pago').isNotNull()).count()}")
print(f"Filas con metodo_pago nulo:    {silver_evolucionado.filter(col('metodo_pago').isNull()).count()}")
print()
print("💡 Delta infirió 'null' para los registros anteriores")
print("   Sin reescribir ningún archivo Parquet existente")


---
## 💡 Retos — Para quien termina antes

*Estos ejercicios van más allá del contenido del slide y son opcionales.*


In [ ]:
# 💡 RETO 1 — Encontrar el "punto óptimo" de particiones
# Genera una curva de rendimiento y visualízala

import matplotlib.pyplot as plt

particiones_test = [1, 2, 4, 8, 16, 32, 64]
tiempos = []

for n in particiones_test:
    df_test = df_raw.repartition(n)
    t0 = time.time()
    df_test.groupBy("region", "categoria", "canal")            .agg(sum("total"), count("*"))            .count()
    tiempos.append(round(time.time() - t0, 3))
    print(f"  {n:3d} particiones: {tiempos[-1]}s")

plt.figure(figsize=(10, 5))
plt.plot(particiones_test, tiempos, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Número de particiones')
plt.ylabel('Tiempo (segundos)')
plt.title('Impacto del particionamiento en Colab (dataset 10K filas)')
plt.axvline(x=8, color='r', linestyle='--', label='Particiones actuales (8)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/partitioning_curve.png', dpi=100)
plt.show()
print("\n¿En qué punto la curva toca el mínimo?")
print("¿Coincide con el número de cores de Colab?")
print(f"Cores disponibles: {spark.sparkContext.defaultParallelism}")


In [ ]:
# 💡 RETO 2 — Simular el "small files problem" y medirlo
# Escribir con muchas particiones pequeñas vs. pocas particiones grandes

import glob

# Muchos archivos pequeños
df_small = df_raw.repartition(100)
df_small.write.mode("overwrite").parquet("/tmp/small_files")
n_small = len([f for f in glob.glob("/tmp/small_files/*.parquet")])

# Pocos archivos grandes
df_big = df_raw.coalesce(4)
df_big.write.mode("overwrite").parquet("/tmp/big_files")
n_big = len([f for f in glob.glob("/tmp/big_files/*.parquet")])

# Medir tiempo de lectura
t0 = time.time()
spark.read.parquet("/tmp/small_files").count()
t_small = round(time.time() - t0, 3)

t0 = time.time()
spark.read.parquet("/tmp/big_files").count()
t_big = round(time.time() - t0, 3)

print(f"{'':30} {'Archivos':<15} {'Tiempo lectura'}")
print("="*55)
print(f"{'100 particiones (small files)':<30} {n_small:<15} {t_small}s")
print(f"{'4 particiones (big files)':<30} {n_big:<15} {t_big}s")
print()
print("💡 En S3 real, el overhead de listar 100 archivos vs 4")
print("   es mucho más notable (operaciones de API de S3)")
print("   → Por eso OPTIMIZE de Delta Lake existe")


In [ ]:
# 💡 RETO 3 — Implementar el MERGE (ingesta incremental)
# Este es el código real del Lab 1b que verán en EMR
# En Colab funciona con la misma lógica

# Simular nuevos pedidos que llegan (algunos nuevos, algunos son updates)
nuevos_pedidos = spark.createDataFrame([
    # Pedido completamente nuevo
    ("ORD-2026-99999001", "2026-08-10", "BOGOTÁ", "Ropa",
     "Producto nuevo 1", 2, 250000.0, 500000.0, "web", False, "2026-08-10", None),
    # Pedido que ya existía — ahora fue devuelto (update)
    ("ORD-2026-00000001", "2026-08-10", "BOGOTÁ", "Electrónica",
     "Producto update", 1, 100000.0, 100000.0, "app_movil", True, "2026-08-10", None),
], ["pedido_id", "fecha", "region", "categoria", "producto",
    "cantidad", "precio_unit", "total", "canal", "devuelto",
    "_ingested_at", "metodo_pago"])

# MERGE: upsert incremental con garantías ACID
silver_table = DeltaTable.forPath(spark, SILVER_PATH)
(silver_table.alias("existente")
    .merge(
        nuevos_pedidos.alias("nuevo"),
        "existente.pedido_id = nuevo.pedido_id"
    )
    .whenMatchedUpdateAll()        # actualiza si ya existe
    .whenNotMatchedInsertAll()     # inserta si es nuevo
    .execute())

print("✅ MERGE ejecutado")
print()
print("=== Plan del MERGE — busca los Exchange ===")
# El MERGE interno hace un join → WIDE dependency
nuevos_pedidos.join(
    spark.read.format("delta").load(SILVER_PATH),
    "pedido_id", "outer"
).explain()

# Verificar el resultado
silver_post = spark.read.format("delta").load(SILVER_PATH)
devuelto = silver_post.filter(col("pedido_id") == "ORD-2026-00000001").select("pedido_id", "devuelto")
print("\nPedido ORD-2026-00000001 después del MERGE:")
devuelto.show()


---
## ✅ Cierre de la sesión

### Las 3 ideas clave de hoy

| # | Concepto | Lo que aprendiste |
|---|----------|-------------------|
| 1 | **Narrow vs Wide** | Cada `Exchange` en el plan = un shuffle = datos cruzando la red. Identifícalos antes de ejecutar. |
| 2 | **Catalyst Optimizer** | `.explain()` muestra lo que Spark *realmente* ejecuta — no siempre es lo que escribiste. |
| 3 | **Tu primer pipeline** | Bronze → Silver (MERGE Delta) → Gold → KPIs. El ciclo completo sobre datos con errores reales. |

---

### Autoevaluación — ¿Qué puedo hacer después de este notebook?

- [ ] Clasificar una operación como narrow o wide **antes de ejecutarla**
- [ ] Leer `.explain()` e identificar los `Exchange`
- [ ] Explicar por qué `coalesce` no puede aumentar particiones
- [ ] Escribir el pipeline Bronze → Silver con las transformaciones correctas
- [ ] Decir en qué versión de Delta estaba la tabla hace N commits

---

### Para el Lab 1b en casa
El código de este notebook es la base exacta del Lab 1b.
Diferencias en EMR:
- El path `s3://tu-bucket/...` en vez de `/tmp/...`
- `spark.sql.shuffle.partitions` → ajustar a `32` (4 executors × 8 cores)
- El MERGE real sobre 500K filas mostrará un DAG más complejo y tiempos reales

---

### Quiz 2 — preparación
El Quiz 2 abre S6. Cubre:
- Particiones de Kafka, offsets, consumer groups
- Garantías de entrega: at-most-once, at-least-once, exactly-once
- Por qué Kafka es **AP** en el Teorema CAP

**Lecturas obligatorias:**
1. Kleppmann DDIA, cap. 11 (Stream Processing) — ~60 min
2. kafka.apache.org/documentation → Introduction + Design — ~45 min
